# Task 1: SU2 on Colab, first light, oblique-shock validation, timing

Runs top to bottom on a CPU runtime. Every cell skips work that is already done; set `FORCE = True`
in the first code cell to redo downloads and runs. Results land in `su2queue/runs/task1/` on Drive.

## 0. Environment
Prints core count, RAM and OS. Look at: `nproc` (expect 2) and the total memory (expect about 12 GB).

In [ ]:
FORCE = False   # True: redo downloads and runs even if their outputs exist
!nproc
!free -h
!head -2 /etc/os-release

## 1. Repository
Clones the repo, checks out `COMMIT` (a branch, tag or sha), installs it, prints the exact commit.
Look at: the printed sha. It goes into the README and every `job.json`.
After pulling a new commit, restart the runtime (Runtime -> Restart runtime) before running: Python
keeps the old modules in memory otherwise. The cell refuses to run if repo modules are already loaded.


In [ ]:
COMMIT = "main"   # branch, tag or commit sha
import os, sys, subprocess
if not os.path.isdir("/content/ai-engineering"):
    !git clone -q https://github.com/emarinhoss/ai-engineering.git /content/ai-engineering
%cd /content/ai-engineering
# Modules already imported by this kernel would not see a new commit: refuse to continue.
stale = [m for m in sys.modules if m.split(".")[0] in ("physics", "geometry", "cases", "postprocess", "worker", "mcp_server")]
assert not stale, f"repo modules already imported in this kernel ({stale[:3]}...): Runtime -> Restart runtime, then run again"
!git fetch -q --all
!git checkout -q {COMMIT}
!git pull -q --ff-only
!pip install -q -e .
sys.path.insert(0, "/content/ai-engineering")
HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("commit", HEAD)
assert HEAD == subprocess.check_output(["git", "rev-parse", f"origin/{COMMIT}"], text=True).strip(), "local checkout is not at the remote commit"


## 2. SU2 v8.2.0 binaries (serial and MPI)
Downloads both release variants and unzips them (each zip contains a second zip with `bin/`), installs OpenMPI only if `mpirun` is missing, records the
sha256 of each zip and the first line of `SU2_CFD --help`. Look at: the two version lines in the
printed JSON (they must say 8.2.0). Colab runs as root, hence `--allow-run-as-root`.

In [ ]:
import hashlib, json, shutil
os.environ.update(OMPI_ALLOW_RUN_AS_ROOT="1", OMPI_ALLOW_RUN_AS_ROOT_CONFIRM="1")
if not shutil.which("mpirun"):
    !apt-get -qq install -y openmpi-bin
REL = "https://github.com/su2code/SU2/releases/download/v8.2.0"
SU2 = {"serial": ("SU2-v8.2.0-linux64.zip", "/content/su2"),
       "mpi": ("SU2-v8.2.0-linux64-mpi.zip", "/content/su2mpi")}
versions = {}
for kind, (zipname, root) in SU2.items():
    zp = f"/content/{zipname}"
    if FORCE or not os.path.exists(zp):
        !wget -q -O {zp} {REL}/{zipname}
    if FORCE or not os.path.exists(f"{root}/bin/SU2_CFD"):
        # The release zip holds a second zip (linux64.zip) that holds bin/; unzip both.
        !mkdir -p {root} && unzip -q -o {zp} -d {root}
        for inner in [f for f in os.listdir(root) if f.endswith(".zip")]:
            !unzip -q -o {root}/{inner} -d {root}
    assert os.path.exists(f"{root}/bin/SU2_CFD"), f"no bin/SU2_CFD in {root}; run !find {root} -name SU2_CFD"
    !chmod +x {root}/bin/*
    out = subprocess.run([f"{root}/bin/SU2_CFD", "--help"], capture_output=True, text=True, timeout=60, check=False)
    first = next((l.strip() for l in (out.stdout + out.stderr).splitlines() if l.strip()), "")
    versions[kind] = {"zip": zipname, "sha256": hashlib.sha256(open(zp, "rb").read()).hexdigest(),
                      "help_first_line": first}
json.dump(versions, open("/content/su2_versions.json", "w"), indent=1)
print(json.dumps(versions, indent=1))
SU2_SERIAL = "/content/su2/bin/SU2_CFD"
SU2_MPI = "mpirun -n 2 --allow-run-as-root /content/su2mpi/bin/SU2_CFD"

## 3. Google Drive queue folder
Mounts Drive and checks that `su2queue/jobs` and `su2queue/runs` exist. Task 1 results go to
`runs/task1`; cases run on the local disk (`/content/work`) and are copied to Drive when finished,
so Drive latency does not enter the timings.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
QUEUE = "/content/drive/MyDrive/su2queue"
for d in ("jobs", "runs"):
    assert os.path.isdir(f"{QUEUE}/{d}"), f"{QUEUE}/{d} is missing"
T1 = f"{QUEUE}/runs/task1"
WORK = "/content/work"
os.makedirs(T1, exist_ok=True); os.makedirs(WORK, exist_ok=True)

## 4. First light (SU2 Tutorials inviscid wedge)
Mach 2 tutorial case from the su2code/Tutorials repository (LGPL), 300 iterations, serial binary.
It only proves the binary runs. No physics claim. Look at: `exit_code` 0, the wall time, and
"Exit Success" at the end of the log.

In [ ]:
import time
FL = "/content/firstlight"
TUT = "https://raw.githubusercontent.com/su2code/Tutorials/master/compressible_flow/Inviscid_Wedge"
os.makedirs(FL, exist_ok=True)
for f in ("mesh_wedge_inv.su2", "inv_wedge_HLLC.cfg"):
    if FORCE or not os.path.exists(f"{FL}/{f}"):
        !wget -q -O {FL}/{f} {TUT}/{f}
!sed -i 's/^\(EXT_\)\?ITER *=.*/ITER= 300/' {FL}/inv_wedge_HLLC.cfg
fl_json = f"{T1}/firstlight.json"
if FORCE or not os.path.exists(fl_json):
    t0 = time.time()
    with open(f"{FL}/log.txt", "w") as f:
        r = subprocess.run([SU2_SERIAL, "inv_wedge_HLLC.cfg"], cwd=FL, stdout=f, stderr=subprocess.STDOUT, check=False)
    json.dump({"exit_code": r.returncode, "wall_time_s": time.time() - t0, "iterations": 300}, open(fl_json, "w"))
firstlight = json.load(open(fl_json))
print(firstlight)
!tail -n 12 {FL}/log.txt 2>/dev/null

## 5. V1: Euler wedge, 10 degrees, medium mesh
`worker.make_case.make` builds mesh and config, `cases.check_keys` compares the config keys with the
SU2 v8.2.0 template, `worker.run_case` runs (2 processes, serial if MPI does not start), then
`oblique_shock_check` compares beta, p2/p1 and M2 with the analytic values.
Look at: no unknown keys, convergence status, and the PASS/FAIL table. A FAIL is reported, not tuned.

In [ ]:
from worker.make_case import make, case_id
from worker.run_case import run_first_working
CASES = {}

def get_case(params, cmds):
    """Build and run a case on local disk, copy it to Drive; reuse earlier results unless FORCE."""
    cid = case_id(params); wk, dr = f"{WORK}/{cid}", f"{T1}/{cid}"
    CASES[cid] = params
    if not FORCE and not os.path.exists(f"{wk}/result_partial.json") and os.path.exists(f"{dr}/result_partial.json"):
        shutil.copytree(dr, wk, dirs_exist_ok=True)
    if FORCE or not os.path.exists(f"{wk}/result_partial.json"):
        shutil.rmtree(wk, ignore_errors=True)
        make(params, wk)
        run_first_working(wk, cmds, timeout_s=2400)
        shutil.copytree(wk, dr, dirs_exist_ok=True)
    return cid, wk, json.load(open(f"{wk}/result_partial.json"))

p_euler = {"family": "wedge", "theta1_deg": 10.0, "solver": "EULER", "mesh_level": "medium",
           "save_volume": True, "tag": "task1", "max_iter": 4000}
cid_e, wk_e, res_e = get_case(p_euler, [SU2_MPI, SU2_SERIAL])
print(cid_e, res_e["solver"]["command"], "wall", round(res_e["run"]["wall_time_s"], 1), "s")
!python -m cases.check_keys {wk_e}/case.cfg cases/su2_v8.2.0_config_template.cfg
print(json.dumps(res_e["history_summary"], indent=1))
from postprocess.validation import oblique_shock_check
v1 = oblique_shock_check(wk_e, 10.0)
d = v1["details"]
if "errors" not in d:
    print("V1 could not be evaluated:", d)
else:
    print(f"{'quantity':10s} {'CFD':>9s} {'analytic':>9s} {'error':>10s}  tolerance")
    print(f"{'beta deg':10s} {v1['beta_deg']:9.3f} {v1['beta_ref_deg']:9.3f} {d['errors']['beta_deg']:+8.3f} deg {d['tol_beta_deg']} deg")
    print(f"{'p2/p1':10s} {v1['p_ratio']:9.4f} {v1['p_ratio_ref']:9.4f} {100*d['errors']['p_ratio_rel']:+8.2f} %  {100*d['tol_ratio']:.0f} %")
    print(f"{'M2':10s} {v1['M2']:9.4f} {v1['M2_ref']:9.4f} {100*d['errors']['M2_rel']:+8.2f} %  {100*d['tol_ratio']:.0f} %")
print("V1", "PASS" if v1["pass"] else "FAIL")
json.dump(v1, open(f"{T1}/v1_oblique_shock.json", "w"), indent=1, default=float)

## 6. V2: laminar wedge timing, heat-flux sign, V2a
Same wedge with `NAVIER_STOKES` (isothermal 300 K) to convergence on 2 processes. Then two 200-iteration
runs, serial and 2 processes, for a like-for-like speed-up (200 iterations include start-up, so their
s/iteration is biased high; the budget uses the long run). Heat-flux sign: trapezoid integral of the
`Heat_Flux` column along the body; for a cold wall the heat goes into the wall, so a positive value
means SU2 counts heat into the wall as positive. V2a reuses `oblique_shock_check` on the laminar
field (its plateau window starts 1 mm above the wall, V2a asks for 3 mm: a small difference);
expected plateau +0 to +6 percent over inviscid. V2a is a report, not a gate.

In [ ]:
p_lam = {**p_euler, "solver": "NAVIER_STOKES", "max_iter": 6000}
cid_l, wk_l, res_l = get_case(p_lam, [SU2_MPI, SU2_SERIAL])
cid_m, wk_m, res_m = get_case({**p_lam, "max_iter": 200, "tag": "task1-mpi200"}, [SU2_MPI, SU2_SERIAL])
cid_s, wk_s, res_s = get_case({**p_lam, "max_iter": 200, "tag": "task1-serial"}, [SU2_SERIAL])
cells = json.load(open(f"{wk_l}/mesh_markers.json"))["cells"]
per_iter = lambda r: r["run"]["wall_time_s"] / max(1, r["run"]["iterations"])
for label, r in (("laminar full", res_l), ("200 it mpi", res_m), ("200 it serial", res_s)):
    s = per_iter(r)
    print(f"{label:14s} cores={r['solver']['n_cores']} it={r['run']['iterations']:5d} wall={r['run']['wall_time_s']:8.1f} s"
          f"  {s:.4f} s/it  {s / (cells / 1000):.5f} s/it per 1000 cells")
print("speed-up serial/mpi (200 it):", round(per_iter(res_s) / per_iter(res_m), 2), "cells", cells)
print(json.dumps(res_l["history_summary"], indent=1))

import numpy as np
from postprocess.tecplot import read_tecplot_ascii, surface_nodes_for_marker
surf = read_tecplot_ascii(f"{wk_l}/surface_flow.dat")
heat_flux = {"case_id": cid_l, "su2_version": res_l["solver"]["version"], "columns": surf["variables"]}
if "Heat_Flux" not in surf["variables"]:
    print("no Heat_Flux column; available:", surf["variables"])
else:
    body = surface_nodes_for_marker(surf, f"{wk_l}/mesh_markers.json", "body")
    body = body[np.argsort(surf["data"]["x"][body])]
    x, q = surf["data"]["x"][body], surf["data"]["Heat_Flux"][body]
    print("rows used:", len(body), "body nodes:", len(json.load(open(f"{wk_l}/mesh_markers.json"))["body"]))
    integral = float(np.sum(0.5 * (q[1:] + q[:-1]) * np.diff(x)))
    heat_flux.update(rows=len(body), sum=float(q.sum()), integral_W_per_m=integral,
                     positive_into_wall_for_cold_wall=integral > 0, multiply_by_for_into_wall=1 if integral > 0 else -1)
    print("sum", q.sum(), "integral over body [W/m]", integral)
    print("Heat_Flux > 0 means heat into the wall for the cold wall:", integral > 0)
json.dump(heat_flux, open(f"{T1}/heat_flux_sign.json", "w"), indent=1)

v2a = oblique_shock_check(wk_l, 10.0)
v2a_pct = None
if "s2" in v2a["details"] and "s2" in v1["details"]:
    v2a_pct = 100 * (v2a["details"]["s2"]["plateau"] / v1["details"]["s2"]["plateau"] - 1)
    print(f"V2a (report): laminar plateau at x = 0.08 is {v2a_pct:+.2f} % relative to inviscid;"
          f" expected 0 to +6 %: {'in range' if 0 <= v2a_pct <= 6 else 'INVESTIGATE'}")
else:
    print("V2a could not be evaluated:", v2a["details"])
json.dump({"plateau_pct_vs_inviscid": v2a_pct, "check": v2a}, open(f"{T1}/v2a.json", "w"), indent=1, default=float)

## 7. Budget and gate T1
From the laminar run: iterations until the density residual is 6 orders below its initial level (or
the iteration cap, flagged), seconds per iteration, cells. Estimated seconds for a 25 000-cell mesh =
s/it x (25000 / cells) x iterations (assumes cost proportional to cells). Verdict per `00_decisions.md`
section 5: <= 5 min go (pool 96); 5 to 10 min pool 64 and Colab Pro; > 10 min coarse mesh is production.
`CONV_RESIDUAL_MINVAL` for production = the initial log10 level minus 6 (printed). Look at: `capped`
(if True the run never reached minus 6 and the estimate is a lower bound) and the verdict.

In [ ]:
from postprocess.history import load, find_column
hs = res_l["history_summary"]
cols = load(f"{wk_l}/history.csv")
rms = cols[find_column(cols, "rms[rho]")]
start = hs["rms_rho_log10_start"]                   # max of the first 50 iterations
hit = np.where(rms <= start - 6)[0]
hit = hit[hit >= 200]                               # CONV_STARTITER
capped = len(hit) == 0
iters = res_l["run"]["iterations"] if capped else int(hit[0]) + 1
s_it = per_iter(res_l)
est_s = s_it * (25000 / cells) * iters
verdict = ("GO: medium mesh, pool 96" if est_s <= 300 else
           "pool 64 and Colab Pro" if est_s <= 600 else "coarse mesh as production")
budget = {"laminar_case_id": cid_l, "status": hs["status"], "drop_orders": hs["drop_orders"],
          "iterations_run": res_l["run"]["iterations"], "iterations_to_minus6": None if capped else iters,
          "capped": capped, "cells": cells, "n_cores": res_l["solver"]["n_cores"], "s_per_iter": s_it,
          "s_per_iter_per_1000_cells": s_it / (cells / 1000), "s_per_iter_serial_200": per_iter(res_s),
          "s_per_iter_mpi_200": per_iter(res_m), "est_s_25k_cells": est_s, "est_min_25k_cells": est_s / 60,
          "runs_per_hour": 3600 / est_s, "gate_T1": verdict,
          "rms_rho_log10_start": start, "CONV_RESIDUAL_MINVAL_suggested": round(start - 6, 1)}
json.dump(budget, open(f"{T1}/budget.json", "w"), indent=1)
print(json.dumps(budget, indent=1))

readme = [
    "# Task 1 results (generated by notebooks/00_su2_colab.ipynb)", "",
    f"- code commit: {HEAD}",
    f"- SU2 serial: {versions['serial']['help_first_line']} (sha256 {versions['serial']['sha256']})",
    f"- SU2 mpi: {versions['mpi']['help_first_line']} (sha256 {versions['mpi']['sha256']})",
    f"- runtime: {res_l['run']['runtime']}",
    f"- first light (tutorial wedge, 300 it, serial): exit {firstlight['exit_code']}, {firstlight['wall_time_s']:.1f} s",
    f"- V1 Euler wedge {cid_e}: {'PASS' if v1['pass'] else 'FAIL'}; beta {v1['beta_deg']:.3f} deg (ref {v1['beta_ref_deg']:.3f}),"
    f" p2/p1 {v1['p_ratio']:.4f} (ref {v1['p_ratio_ref']:.4f}), M2 {v1['M2']:.4f} (ref {v1['M2_ref']:.4f}); status {res_e['history_summary']['status']}",
    f"- laminar wedge {cid_l}: status {hs['status']}, drop {hs['drop_orders']:.2f} orders, {res_l['run']['iterations']} iterations,"
    f" {res_l['run']['wall_time_s']:.0f} s on {res_l['solver']['n_cores']} process(es), {cells} cells",
    f"- V2a (report): plateau vs inviscid {v2a_pct if v2a_pct is None else format(v2a_pct, '+.2f')} % (expected 0 to +6)",
    f"- heat flux sign: integral over body {heat_flux.get('integral_W_per_m')} W/m; positive into the wall for a cold wall: {heat_flux.get('positive_into_wall_for_cold_wall')}",
    f"- seconds per iteration per 1000 cells: {s_it / (cells / 1000):.5f} (full run); 200-iteration runs: mpi {per_iter(res_m) / (cells / 1000):.5f} ({cid_m}), serial {per_iter(res_s) / (cells / 1000):.5f} ({cid_s})",
    f"- initial log10 residual {start:.2f}; CONV_RESIDUAL_MINVAL = {start - 6:.1f}",
    f"- iterations used for the estimate: {iters}{' (CAPPED: residual never reached minus 6)' if capped else ''}",
    f"- 25k-cell estimate: {est_s:.0f} s per run, {3600 / est_s:.1f} runs per hour",
    f"- GATE T1: {verdict}",
    f"- case ids: euler {cid_e}, laminar {cid_l}, mpi200 {cid_m}, serial200 {cid_s}",
]
open(f"{T1}/README.md", "w").write("\n".join(readme) + "\n")
print("\n".join(readme))

## 8. Records
Gzips `surface_flow.dat` and `log.txt` in the Drive copies of each case (`flow.dat` stays
uncompressed), then lists the folder with sizes and prints the README.

In [ ]:
for cid in CASES:
    for name in ("surface_flow.dat", "log.txt"):
        if os.path.exists(f"{T1}/{cid}/{name}"):
            !gzip -f {T1}/{cid}/{name}
!ls -lhR {T1}
print(open(f"{T1}/README.md").read())

## What to report back
Paste `su2queue/runs/task1/README.md` (printed above) into the session. It holds the SU2 version string,
the V1 table, the laminar convergence status, V2a, the heat-flux sign, seconds per iteration per
thousand cells for 1 and 2 processes, the 25k-cell estimate and the gate T1 verdict, with every case id.
Also report anything that failed (unknown config keys, MPI falling back to serial, V1 FAIL).